In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
import numpy as np
import string
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
print(df.head())
print(df.columns.tolist())

   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   
3   4  Select the most accurate option: What is Marti...   
4   5  Identify the correct statement: What is the co...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   
3  Martin Heidegger believes that humans exist wi...   
4  Simultaneity is relative, meaning that two eve...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   
2                                        Redshifting   
3  Martin Heidegger believes that humans do not e...   
4  Simultaneity is rel

In [3]:
# Q1: Frequency distribution of correct answers
freq = df['answer'].value_counts()
print(freq)
print("Most frequent:", freq.max(), "Least frequent:", freq.min())
print("Sum:", freq.max() + freq.min())

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Most frequent: 490 Least frequent: 324
Sum: 814


In [4]:
# Q2: Vocabulary size after cleaning
def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

df['clean_prompt'] = df['prompt'].apply(clean_text)
all_words = ' '.join(df['clean_prompt']).split()
vocab = set(all_words)
print("Vocabulary size:", len(vocab))

Vocabulary size: 859


In [5]:
# Q3: Words left in Row ID 1 after removing stop words
row1 = df[df['id'] == 1]['clean_prompt'].values[0]
words = row1.split()
filtered = [w for w in words if w not in ENGLISH_STOP_WORDS]
print("Words in Row ID 1 after stop word removal:", len(filtered))
print(filtered)

Words in Row ID 1 after stop word removal: 13
['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']


In [6]:
# Q4: TF-IDF vectorizer on prompts + all options
options = ['A', 'B', 'C', 'D', 'E']

def get_combined_texts(df):
    texts = []
    for _, row in df.iterrows():
        texts.append(str(row['prompt']))
        for opt in options:
            texts.append(str(row[opt]))
    return texts

all_texts = get_combined_texts(df)
vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(all_texts)
print("Vocabulary size (TF-IDF):", len(vectorizer.vocabulary_))

Vocabulary size (TF-IDF): 2762


In [7]:
# Q5: Cosine similarity between prompt and option A for Row ID 1
row1 = df[df['id'] == 1].iloc[0]
prompt_vec = vectorizer.transform([row1['prompt']])
optA_vec = vectorizer.transform([row1['A']])
sim = cosine_similarity(prompt_vec, optA_vec)[0][0]
print(f"Cosine similarity (prompt vs A) for Row ID 1: {sim:.4f}")

Cosine similarity (prompt vs A) for Row ID 1: 0.2328


In [8]:
# Q6: % where highest cosine similarity option matches correct answer
def get_best_option(row):
    prompt_vec = vectorizer.transform([str(row['prompt'])])
    sims = {}
    for opt in options:
        opt_vec = vectorizer.transform([str(row[opt])])
        sims[opt] = cosine_similarity(prompt_vec, opt_vec)[0][0]
    return max(sims, key=sims.get)

df['predicted'] = df.apply(get_best_option, axis=1)
accuracy = (df['predicted'] == df['answer']).mean() * 100
print(f"Accuracy (highest cosine = correct): {accuracy:.1f}%")

Accuracy (highest cosine = correct): 13.7%


In [9]:
# Q7: MAP@3 if ground truth=C, prediction=[C,A,B]
def map_at_3(truth, preds):
    score = 0.0
    hits = 0
    for i, p in enumerate(preds[:3], 1):
        if p == truth:
            hits += 1
            score += hits / i
    return score / min(1, 1)  # only 1 relevant item

print("MAP@3 (truth=C, preds=[C,A,B]):", map_at_3('C', ['C','A','B']))

MAP@3 (truth=C, preds=[C,A,B]): 1.0


In [10]:
# Q8: MAP@3 if ground truth=B, prediction=[D,B,E]
print("MAP@3 (truth=B, preds=[D,B,E]):", map_at_3('B', ['D','B','E']))

MAP@3 (truth=B, preds=[D,B,E]): 0.5


In [11]:
# Q9: Majority Class Baseline MAP@3
freq_order = df['answer'].value_counts().index.tolist()  # sorted by frequency
top3 = freq_order[:3]
print("Top 3 majority classes:", top3)

def map_at_k(truth, preds, k=3):
    score = 0.0
    hits = 0
    for i, p in enumerate(preds[:k], 1):
        if p == truth:
            hits += 1
            score += hits / i
    return score

scores = df['answer'].apply(lambda x: map_at_k(x, top3))
print(f"Majority Class Baseline MAP@3: {scores.mean():.4f}")

Top 3 majority classes: ['B', 'C', 'A']
Majority Class Baseline MAP@3: 0.4213


In [12]:
# Q10: TF-IDF Pipeline MAP@3
def get_top3_options(row):
    prompt_vec = vectorizer.transform([str(row['prompt'])])
    sims = {}
    for opt in options:
        opt_vec = vectorizer.transform([str(row[opt])])
        sims[opt] = cosine_similarity(prompt_vec, opt_vec)[0][0]
    sorted_opts = sorted(sims, key=sims.get, reverse=True)
    return sorted_opts[:3]

df['top3_preds'] = df.apply(get_top3_options, axis=1)
tfidf_scores = df.apply(lambda row: map_at_k(row['answer'], row['top3_preds']), axis=1)
print(f"TF-IDF Pipeline MAP@3: {tfidf_scores.mean():.4f}")

TF-IDF Pipeline MAP@3: 0.3119
